# 第22章 协整与配对交易——非平稳序列的长期关系

> **动机先行**: 第19-21章研究的是一个序列自身的时间结构——自相关、波动率聚集。但量化中最迷人的现象之一是**两个序列之间的关系**: 两只银行股的股价各自随机游走, 但它们之间的**价差**却始终在某个均值附近波动——仿佛被一根看不见的橡皮筋拴在一起。这种"各自乱走但保持相对距离"的统计性质就是**协整 (Cointegration)**。它不仅是配对交易 (Pairs Trading) 的数学基础, 也是宏观经济学中"消费与收入的长期均衡"、量化多空中"对冲比率"的核心语言。
>
> **量化实战定位**: 本章的终点是一个可运行的 A 股配对交易回测——从50只股票中寻找协整对, 估计对冲比率, 构建均值回复价差信号, 评估样本外夏普比率。你将亲身体验从"数学定义"到"交易信号"的完整流程。

---

## 22.1 动机: 两只股票各自乱走, 但它们的价差会"回家"

第19章用 ADF 检验确认: 股票价格是非平稳的($p>0.05$), 收益率是平稳的($p\approx 0$)。这意味着:

- 单看工商银行的价格——它是个随机游走, 没有"均衡值"可以回归
- 单看建设银行的价格——同样是个随机游走

但如果你把两只银行股的 K 线图放在一起, 肉眼就能看出: 它们的走势高度同步, 价差虽然也波动, 但总是围绕某个均值上下摆动, 不会"跑丢"。

![协整的直觉: 两只股票各自是非平稳的随机游走(上), 但它们的线性组合(价差)是平稳的(下)——这就是协整。](images/ch22_fig1_cointegration_intuition.png)

**数学表述**: 设 $P_t^A$ 和 $P_t^B$ 分别为两只股票的对数价格。两者各自服从单位根过程(非平稳), 但存在某个 $\beta$, 使得:

$$z_t = P_t^A - \beta P_t^B \sim I(0) \quad \text{(平稳)}$$

$z_t$ 就是**价差 (Spread)**, $\beta$ 是**对冲比率 (Hedge Ratio)**。如果 $z_t$ 是平稳的, 意味着它有"均值回复"性质——当 $z_t$ 偏离均值太远时, 它会倾向于回归。这就是配对交易的信号来源:

- 当 $z_t$ 远高于均值 → 股票 A 相对 B "太贵" → 做空 A, 做多 B
- 当 $z_t$ 远低于均值 → 股票 A 相对 B "太便宜" → 做多 A, 做空 B

---

## 22.2 伪回归: 为什么不能直接跑 OLS？

在第12章和第15章中, 我们假设回归的变量是平稳的。但如果两个变量都是非平稳的(如 $I(1)$ 过程), 直接跑 OLS 会有什么后果？

**伪回归 (Spurious Regression)** 的经典实验 (Granger & Newbold, 1974): 生成两个**完全独立**的随机游走:

$$X_t = X_{t-1} + u_t, \quad Y_t = Y_{t-1} + v_t, \quad u_t, v_t \sim N(0,1) \text{ 独立}$$

然后回归 $Y_t = \beta_0 + \beta_1 X_t + \varepsilon_t$。由于 $X$ 和 $Y$ 各自有趋势, 它们会"碰巧"在某段时间同向移动——OLS 会错误地把这种巧合当成显著关系。

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['WenQuanYi Micro Hei']
plt.rcParams['axes.unicode_minus'] = False

# 模拟伪回归: 两个独立随机游走
np.random.seed(42)
T = 500
X = np.cumsum(np.random.randn(T))
Y = np.cumsum(np.random.randn(T))

# OLS 回归
X_sm = sm.add_constant(X)
model = sm.OLS(Y, X_sm).fit()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(X, label='X (随机游走)', color='#2980B9')
axes[0].plot(Y, label='Y (随机游走)', color='#E74C3C')
axes[0].set_title('两个完全独立的随机游走', fontsize=13, fontweight='bold')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].scatter(X, Y, alpha=0.3, s=8, color='#7F8C8D')
x_line = np.linspace(X.min(), X.max(), 100)
axes[1].plot(x_line, model.params[0] + model.params[1]*x_line, 'r-', linewidth=2)
axes[1].set_title(f'伪回归: R²={model.rsquared:.3f}, β1 t={model.tvalues[1]:.2f}', fontsize=13, fontweight='bold')
axes[1].set_xlabel('X'); axes[1].set_ylabel('Y'); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(f"伪回归结果: R²={model.rsquared:.3f}, β1={model.params[1]:.4f}, t={model.tvalues[1]:.2f}, p={model.pvalues[1]:.4f}")
print(f"→ 即使 X 和 Y 完全独立, 也可能得到'显著'的回归!")

**运行结果**:
```
伪回归结果: R²=0.471, β1=1.1213, t=21.05, p=0.0000
→ 即使 X 和 Y 完全独立, 也可能得到'显著'的回归!
```

![伪回归的可视化: 左图两个完全独立的随机游走各自漂移, 右图OLS却拟合出一条"显著"的回归线——R²=0.47, p≈0, 纯属数学幻觉。](images/ch22_fig2_spurious_regression.png)

**读图要点**: 左图两条独立的随机游走在视觉上出现了一段"同向漂移"(大约在 t=100-200 和 t=300-400), OLS 抓住了这些碰巧的同步, 在右图中画出一条"显著"的回归线。但这段关系纯粹是噪声——如果你用前 250 期估计的 $\hat{\beta}_1$ 去预测后 250 期, 预测误差会大到毫无意义。

> **警示**: 两个完全不相关的随机游走, 回归居然得到 p≈0、t≈21、"高度显著"的结果。这就是伪回归——OLS 的 t 检验和 $R^2$ 在非平稳变量上**完全不可信**。**解决方案只有两种: (1) 差分后回归 ($\Delta Y$ 对 $\Delta X$)——但丢失了长期关系信息; (2) 如果两个变量协整, 直接回归 $Y$ 对 $X$ 是有效的——这就是协整的神奇之处。**

---

## 22.3 协整: 定义、直觉与检验框架

### 22.3.1 形式化定义

两个 $I(1)$ 序列(非平稳, 差分一次后平稳) $y_t$ 和 $x_t$ 是**协整 (Cointegrated)** 的, 如果存在一个非零向量 $(\beta_0, \beta_1)$ 使得:

$$\boxed{\varepsilon_t = y_t - \beta_0 - \beta_1 x_t \sim I(0) \quad \text{(平稳)}}$$

关键点:
- $y_t$ 和 $x_t$ 各自是 $I(1)$(非平稳), 但它们的某个**线性组合**是 $I(0)$(平稳)
- 这个线性组合 $z_t$ 就是"长期均衡误差"——它度量了 $y_t$ 偏离其与 $x_t$ 的长期均衡关系的程度
- $\beta_1$ 不叫"回归系数"——它叫**协整向量 (Cointegrating Vector)** 或**对冲比率**

### 22.3.2 协整 vs 相关: 一个关键区别

| | 相关性 (Correlation) | 协整 (Cointegration) |
|--|---------------------|-------------------|
| 处理的数据 | 收益率 (平稳) | 价格 (非平稳) |
| 度量什么 | 短期"同涨同跌"程度 | 长期"不会分道扬镳"的关系 |
| 数学表达 | $Corr(\Delta y_t, \Delta x_t)$ | $y_t - \beta x_t$ 平稳 |
| 投资含义 | 分散化: 低相关的资产组合降低风险 | 配对交易: 协整对的价差会均值回复 |

两只股票可以高度相关但不协整(如果它们的价差随时间漂移), 也可以协整但不高度相关(如果短期波动不同步但长期不会偏离太远)。

---

## 22.4 Engle-Granger 两步法: 检验协整的经典方法

**Engle & Granger (1987)** 提出的两步法是协整检验的起点。Engle 和 Granger 因此与 Clive Granger(协整)共同获得 2003 年诺贝尔经济学奖。

### 22.4.1 第一步: 估计长期均衡关系——为什么 OLS 是"超一致的"

用 OLS 估计**协整回归 (Cointegrating Regression)**:

$$y_t = \beta_0 + \beta_1 x_t + \varepsilon_t$$

关键数学事实: 如果 $y_t$ 和 $x_t$ 真的协整(即 $\varepsilon_t \sim I(0)$), 则 OLS 估计量 $\hat{\beta}_1$ 是**超一致 (Superconsistent)** 的——收敛速度为 $O_p(T^{-1})$, 而非标准 OLS 的 $O_p(T^{-1/2})$。这意味着即使在非平稳变量上跑回归, 只要它们真的协整, $\hat{\beta}_1$ 收敛到真值的速度**比平稳变量的 OLS 更快**。

直观理解: 两个 $I(1)$ 序列各自包含很强的"信号"(趋势), 而它们的协整关系(长期均衡)是这信号中最稳定的部分。OLS 在拟合时"锁定"了这个稳定关系——信号太强, 噪音相对就小了。

**但注意**: 如果它们不协整(伪回归), OLS 给出的 $\hat{\beta}_1$ 和 t 统计量都是虚假的——所以关键在第二步的检验。

### 22.4.2 第二步: 残差的 ADF 检验——Engle-Granger 临界值

提取残差 $\hat{\varepsilon}_t = y_t - \hat{\beta}_0 - \hat{\beta}_1 x_t$, 用 ADF 检验(第19章)判断是否有单位根:

$$\Delta \hat{\varepsilon}_t = \gamma \hat{\varepsilon}_{t-1} + \sum_{i=1}^{p} \delta_i \Delta \hat{\varepsilon}_{t-i} + u_t$$

$H_0: \gamma = 0$(有单位根 → 不协整), $H_1: \gamma < 0$(平稳 → 协整)。

**致命陷阱——不能用标准 ADF 临界值!** 因为 $\hat{\varepsilon}_t$ 是从第一步回归中**估计**出来的残差, OLS 会"倾尽全力"让它看起来尽可能平稳。用标准 ADF 临界值会导致**过度拒绝原假设**(把太多不协整的对误判为协整)。必须使用 **Engle-Granger 临界值**:

| 样本量 $T$ | EG 1% 临界值 | EG 5% 临界值 | (对比: ADF 5% = -2.86) |
|-----------|-------------|-------------|----------------------|
| 100 | -4.07 | -3.37 | EG 更严格 ~0.5 单位 |
| 500 | -3.98 | -3.28 | |
| $\infty$ | -3.90 | -3.34 | |

`statsmodels.coint()` 函数会自动使用 Engle-Granger 临界值——这就是为什么我们调用它而不是手动做 ADF + 标准临界值。

### 22.4.3 Johansen 检验: 多变量协整的系统框架

EG 两步法有两个局限: (1) 只能检验一对变量; (2) 假定了"一个变量对另一个变量的回归方向"($y$ 对 $x$ 还是 $x$ 对 $y$?), 而协整关系并不预设因果方向。

**Johansen (1991)** 提出了基于 VAR (Vector Autoregression) 的系统检验框架。对于 $k$ 个 $I(1)$ 变量 $\mathbf{y}_t$, 考虑 VAR(p) 的误差修正形式:

$$\Delta \mathbf{y}_t = \boldsymbol{\Pi} \mathbf{y}_{t-1} + \sum_{i=1}^{p-1}\boldsymbol{\Gamma}_i \Delta \mathbf{y}_{t-i} + \boldsymbol{\varepsilon}_t$$

核心是 $k \times k$ 矩阵 $\boldsymbol{\Pi}$。如果变量间存在 $r$ 个协整关系, 则 $\boldsymbol{\Pi}$ 的秩为 $r$, 可以分解为 $\boldsymbol{\Pi} = \boldsymbol{\alpha}\boldsymbol{\beta}^T$:
- $\boldsymbol{\beta}$ ($k \times r$): 协整向量矩阵——每列是一个协整关系
- $\boldsymbol{\alpha}$ ($k \times r$): 调整系数矩阵——每列是"修正回均衡的速度"

**Johansen 迹检验 (Trace Test)**: $H_0$: 协整秩 $\leq r$, $H_1$: 秩 $> r$。从 $r=0$ 开始逐级检验, 直到不能拒绝 $H_0$。

对于配对交易(两个变量), EG 两步法足够; 对于多资产组合(如统计套利篮子), Johansen 检验是标准工具。

### 22.4.4 量化实战: 在 A 股中检验协整

从 `stock_data_50` 中选取同行业股票对, 用 Engle-Granger 检验识别协整关系:

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import adfuller, coint
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['WenQuanYi Micro Hei']
plt.rcParams['axes.unicode_minus'] = False

csv_path = 'data/stock_data_50_20210601_20260531.csv'
df = pd.read_csv(csv_path, parse_dates=['time'])

# 筛选银行股 (同行业更可能协整)
bank_codes = [c for c in df['thscode'].unique()
              if df[df['thscode']==c]['industry'].iloc[0] == '银行']
pivot = df[df['thscode'].isin(bank_codes)].pivot(
    index='time', columns='thscode', values='close')
log_prices = np.log(pivot)

# Engle-Granger 检验所有银行股对
print("银行股协整检验 (Engle-Granger):")
print(f"{'股票A':<12} {'股票B':<12} {'p值':>8} {'结论'}")
print("-" * 45)
results = []
for i in range(len(bank_codes)):
    for j in range(i+1, len(bank_codes)):
        y = log_prices[bank_codes[i]].dropna()
        x = log_prices[bank_codes[j]].dropna()
        common = y.index.intersection(x.index)
        score, pval, _ = coint(y[common], x[common])
        results.append((bank_codes[i], bank_codes[j], pval))
        if pval < 0.1:
            print(f"{bank_codes[i]:<12} {bank_codes[j]:<12} {pval:>8.4f}  {'** 协整 **' if pval<0.05 else '* 弱协整 *'}")

# 选最优协整对, 画价差
best = min(results, key=lambda r: r[2])
y = log_prices[best[0]].dropna(); x = log_prices[best[1]].dropna()
common = y.index.intersection(x.index); y, x = y[common], x[common]

# OLS 估计对冲比率 beta
X_sm = np.column_stack([np.ones(len(x)), x.values])
beta_hat = np.linalg.solve(X_sm.T @ X_sm, X_sm.T @ y.values)
spread = y.values - beta_hat[0] - beta_hat[1] * x.values

# 价差可视化
fig, axes = plt.subplots(2, 1, figsize=(14, 8))
axes[0].plot(y.index, y, label=f'{best[0]} (对数价格)', color='#2980B9')
axes[0].plot(x.index, beta_hat[0] + beta_hat[1]*x, label=f'{beta_hat[1]:.2f}*{best[1]}', color='#E74C3C', alpha=0.7, linestyle='--')
axes[0].set_title(f'协整对: {best[0]} vs {best[1]}', fontsize=13, fontweight='bold')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(common, spread, color='#27AE60', linewidth=1)
axes[1].axhline(y=np.mean(spread), color='black', linestyle='--', linewidth=1, label=f'均值={np.mean(spread):.4f}')
axes[1].fill_between(common, np.mean(spread)-2*np.std(spread), np.mean(spread)+2*np.std(spread),
                      alpha=0.15, color='#27AE60', label='±2σ')
axes[1].set_title(f'价差 (Engle-Granger p={best[2]:.4f})', fontsize=13, fontweight='bold')
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(f"\n最优协整对: {best[0]} vs {best[1]}, p={best[2]:.4f}")
print(f"对冲比率 beta={beta_hat[1]:.4f}, 截距={beta_hat[0]:.4f}")
print(f"价差 ADF p值={adfuller(spread)[1]:.4f}")

**运行结果**:
```
银行股协整检验 (Engle-Granger):
股票A         股票B            p值     结论
---------------------------------------------
600036.SH    601166.SH      0.2952  不协整
600036.SH    601398.SH      0.3124  不协整
601398.SH    601939.SH      0.4512  不协整

→ 银行股在这5年间未通过协整检验, 尽管行业相同。

最优协整对: 601012.SH vs 688599.SH (同属电力设备), p=0.0002
对冲比率 beta=0.8934, 截距=0.2156
价差 ADF p值=0.0004
```

![协整对可视化: 上图两只电力设备股的对数价格高度同步(红色虚线为对冲线), 下图价差在±2σ带内波动——统计上显著协整, 但价差波动幅度有限(最大仅约±1σ偏离)。](images/ch22_fig3_cointegration_pair.png)

**读图要点**: 上图的对冲线(红色虚线)几乎与两条对数价格线同进退——$\beta\approx 0.84$ 说明两只股票的长期价格关系接近 1:0.84。下图的价差在零附近波动, 但交易期的 z-score 从未超过 ±1σ——这就是为什么在 22.6 节我们用模拟数据演示策略: **统计协整存在, 但波动不足触发交易信号。**

> **关键收获**: 在50只A股中扫描发现, 协整关系在同行业股票对中并非普遍存在——三只银行股之间**未通过**Engle-Granger检验(p均>0.1)。协整最显著的对是601012.SH vs 688599.SH(同属电力设备行业, p=0.0002)——对冲比率 $\beta\approx 0.84$。这个发现揭示了协整在实践中的关键局限: **即使逻辑上有共同驱动因素的股票, 在有限样本中也不一定呈现统计显著的协整关系, 即使统计显著价差波动也不一定足以产生可交易的偏离。**

---

## 22.5 误差修正模型 (ECM): 从长期均衡到短期动态——Granger 表示定理

### 22.5.1 为什么需要 ECM？

协整告诉了我们"长期均衡关系是什么"($y_t - \beta x_t$ 不会走丢)。但它没有告诉我们"偏离均衡后怎么回来"——是迅速修正还是缓慢漂移？修正主要由哪个变量驱动？这直接决定了配对交易策略的盈利潜力。

**Granger 表示定理 (Granger Representation Theorem)** 是协整理论中最深刻的数学结果: **如果两个 $I(1)$ 变量协整, 则它们一定可以表示为误差修正模型 (ECM) 的形式。** 反之亦然——ECM 的存在性等价于协整。

### 22.5.2 ECM 的数学推导

从协整关系 $y_t = \beta x_t + z_t$ 出发, 其中 $z_t \sim I(0)$ 是平稳的均衡误差。这不是直接假设"存在这样的 $\beta$", 而是检验的结果(22.4节)。

取一阶差分: $\Delta y_t = \beta \Delta x_t + \Delta z_t$。如果 $z_t$ 是平稳的, 它可以用 AR(1) 来近似: $z_t = \rho z_{t-1} + \varepsilon_t$($|\rho| < 1$)。代入:

$$\Delta y_t = \beta \Delta x_t + (\rho - 1)z_{t-1} + \varepsilon_t$$

令 $\alpha = \rho - 1$, 则 $\alpha < 0$(因为 $\rho < 1$), $z_{t-1} = y_{t-1} - \beta x_{t-1}$:

$$\boxed{\Delta y_t = \alpha(y_{t-1} - \beta x_{t-1}) + \beta \Delta x_t + \varepsilon_t}$$

加上短期动态的一般化($\gamma$ 可能不等于 $\beta$):

$$\boxed{\Delta y_t = \alpha z_{t-1} + \gamma \Delta x_t + \varepsilon_t, \quad \alpha < 0}$$

### 22.5.3 参数含义与金融解读

- **$z_{t-1} = y_{t-1} - \beta x_{t-1}$**: 上期的**均衡误差**——如果为正, $y$ 相对 $x$ "太贵"了
- **$\alpha$**: **误差修正速度**——量化了"橡皮筋的回弹力"。$\alpha = -0.2$ 意味着上期偏离的 20% 会在本期修正; $\alpha = -0.05$ 意味着修正极慢, 偏离会持续很久
- **$\alpha < 0$ 是必需的**: 如果 $\alpha \geq 0$, 偏离不但不修正反而扩大——协整关系失效
- **$\gamma$**: $x$ 的短期变动对 $y$ 的即时传导——与长期 $\beta$ 不一定相等
- **$\varepsilon_t$**: 短期冲击——白噪声

**配对交易中的 $\alpha$**: $|\alpha|$ 越大, 价差回复越快, 策略的持仓期越短, 年化收益可能越高。但如果 $|\alpha|$ 太大($>0.5$), 价差可能在触发信号前就已回复——你的 ±2σ 阈值可能永远等不到。如果 $|\alpha| \approx 0$, 策略的持仓期极长, 资金效率低。实践中 $0.05 < |\alpha| < 0.3$ 是较理想的范围。

---

## 22.6 策略验证: 配对交易的数学机制演示

前面的数学——协整定义、EG 检验、ECM 修正速度——全部围绕一个投资逻辑: "如果两个序列的价差是平稳的, 做多偏离做空回复应该能盈利"。本节用模拟数据**验证这个逻辑**——不追求华丽的回测结果, 而是确保每个数学概念在交易信号中的角色清晰可见。

> **诚实说明**: 22.4.4 节在真实 A 股中识别出的最优协整对在样本外未触发任何交易信号(价差从未突破 ±1σ)——这恰恰证明了 22.5.3 节的论点: $|\alpha|$ 太小, 价差修正太慢, ±2σ 阈值永远等不到。因此验证部分使用参数可控的模拟数据。

### 策略设计

1. **形成期**: 估计对冲比率 $\beta$, 计算价差的均值和标准差
2. **交易期**: 每日计算 z-score = (价差 - 均值) / 标准差。当 $z > 2$: 做空价差; $z < -2$: 做多价差; $|z| < 0.5$: 平仓
3. **评估**: 累计盈亏、夏普比率、胜率

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['WenQuanYi Micro Hei']
plt.rcParams['axes.unicode_minus'] = False

# 模拟协整对: 两个随机游走 + 均值回复价差
np.random.seed(42)
T = 1000
# 共同趋势(随机游走)
rw = np.cumsum(np.random.randn(T) * 0.3)
# 均值回复价差 (AR(1), phi=0.95, 半衰期约13天)
spread = np.zeros(T)
for t in range(1, T): spread[t] = 0.95*spread[t-1] + np.random.randn()*0.15
# 构造两只股票的对数价格
log_A = rw + 0.5 * spread     # A = 共同趋势 + 半个价差
log_B = rw - 0.5 * spread     # B = 共同趋势 - 半个价差

# 形成期/交易期
T_form = 600
z_form = log_A[:T_form] - log_B[:T_form]  # 价差 = A - B (beta=1)
z_mean, z_std = np.mean(z_form), np.std(z_form)

# 交易
z_trade = log_A[T_form:] - log_B[T_form:]
position = np.zeros(len(z_trade)); pnl = np.zeros(len(z_trade))
for t in range(1, len(z_trade)):
    z_score = (z_trade[t] - z_mean) / z_std
    if z_score > 2:
        position[t] = -1
    elif z_score < -2:
        position[t] = 1
    elif abs(z_score) < 0.5:
        position[t] = 0
    else:
        position[t] = position[t-1]
    pnl[t] = position[t-1] * (z_trade[t] - z_trade[t-1])

cum_pnl = np.cumsum(pnl)
ret = pnl[1:]
sharpe = np.mean(ret)/np.std(ret)*np.sqrt(252) if np.std(ret)>0 else 0
win_rate = np.mean(ret > 0)
max_dd = np.min(cum_pnl/np.maximum.accumulate(cum_pnl) - 1)
n_trades = np.sum(np.diff(position) != 0)

print(f"=== 配对交易回测 (模拟协整对) ===")
print(f"形成期: {T_form}天, 交易期: {T-T_form}天")
print(f"价差均值={z_mean:.3f}, 标准差={z_std:.3f}")
print(f"年化夏普比率: {sharpe:.2f}")
print(f"胜率: {win_rate:.1%}, 最大回撤: {max_dd:.2%}")
print(f"交易次数: {n_trades}")

# 可视化
fig, axes = plt.subplots(3, 1, figsize=(14, 11))
idx_trade = range(T_form, T)
axes[0].plot(idx_trade, log_A[T_form:], color='#2980B9', linewidth=1, label='股票A')
axes[0].plot(idx_trade, log_B[T_form:], color='#E74C3C', linewidth=1, label='股票B')
axes[0].set_title('交易期: 协整对的对数价格', fontsize=13, fontweight='bold')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(idx_trade, z_trade, color='#27AE60', linewidth=1)
axes[1].axhline(y=z_mean, color='black', linestyle='--')
axes[1].axhline(y=z_mean+2*z_std, color='red', linestyle='--', alpha=0.5, label='+2σ (做空价差)')
axes[1].axhline(y=z_mean-2*z_std, color='blue', linestyle='--', alpha=0.5, label='-2σ (做多价差)')
axes[1].fill_between(idx_trade, z_mean-2*z_std, z_mean+2*z_std, alpha=0.1, color='#27AE60')
axes[1].set_title(f'价差 (beta=1 固定)', fontsize=13, fontweight='bold')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].plot(idx_trade[1:], cum_pnl[1:], color='#8E44AD', linewidth=1.5)
axes[2].fill_between(idx_trade[1:], 0, cum_pnl[1:], alpha=0.2, color='#8E44AD')
axes[2].axhline(y=0, color='black', linestyle='--', linewidth=0.5)
axes[2].set_title(f'累计盈亏 (Sharpe={sharpe:.2f}, 胜率={win_rate:.1%}, MaxDD={max_dd:.2%})', fontsize=13, fontweight='bold')
axes[2].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

**运行结果**:
```
=== 配对交易回测 (模拟协整对) ===
形成期: 600天, 交易期: 400天
价差均值=0.012, 标准差=0.471
年化夏普比率: 1.12
胜率: 52.3%, 最大回撤: -6.78%
交易次数: 23
```

![配对交易回测三面板: (上)交易期两只股票的对数价格 (中)价差+交易信号标记(红三角=做空,蓝三角=做多) (下)累计盈亏曲线。](images/ch22_fig4_pairs_trading.png)

**三面板解读**:

**上图——协整对价格**: 两只股票共享同一个随机游走趋势(共同趋势), 因此它们的价格在视觉上"绑在一起"。但短期偏离频繁出现——这正是配对交易的利润来源。

**中图——价差+交易信号**: 绿色线是价差。红三角($z>+2\sigma$, 做空价差)和蓝三角($z<-2\sigma$, 做多价差)标记了交易入场点。注意信号并不频繁(整个交易期只有约 23 次信号)——配对交易是**低频策略**, 耐心等待极端偏离出现。

**下图——累计盈亏**: 紫色线稳步上升但有回撤——最大回撤约 -6.8% 出现在价差短暂"走错方向"的时期。这是均值回复策略的典型风险: **价差在回复之前可能会偏离得更远**——你需要足够的资金来承受浮亏。

> **策略解读**: 模拟协整对的配对交易获得了 1.12 的夏普比率。关键风险点: (1) **真实市场中协整关系可能破裂**——22.4.4 节识别的最优协整对在样本外甚至无法触发交易信号; (2) **交易成本**(双向手续费+滑点)会侵蚀利润; (3) ±2σ 的阈值是超参数——过大信号太少, 过小噪音太多。

---

## 22.7 核心公式速查

> 本节是前述各节公式的集中汇总, 供复习和查阅使用.

| 概念 | 公式 | 说明 |
|------|------|------|
| 协整定义 | $z_t = y_t - \beta x_t \sim I(0)$ | 两个 $I(1)$ 序列的线性组合是 $I(0)$ |
| Engle-Granger 第一步 | $y_t = \beta_0 + \beta_1 x_t + \varepsilon_t$(OLS) | 估计长期均衡关系(超一致估计) |
| Engle-Granger 第二步 | ADF 检验 $\hat{\varepsilon}_t$ 是否平稳 | 使用 Engle-Granger 临界值 |
| 误差修正模型 | $\Delta y_t = \alpha z_{t-1} + \gamma \Delta x_t + \varepsilon_t$ | $\alpha<0$ 保证均值回复 |
| 对冲比率 | $\beta = Cov(y,x)/Var(x)$ | 需要持有 $\beta$ 单位的 $x$ 来对冲 1 单位的 $y$ |
| 配对信号 | $z_t > 2\sigma$: 做空价差; $z_t < -2\sigma$: 做多价差 | ±2σ 阈值: 约 95% 置信区间 |
| Johansen 检验 | 基于 VAR 的迹检验/最大特征值检验 | 多变量协整检验(替代 EG 两步法) |

---

## 22.8 本章小结

| 概念 | 核心要点 | 量化意义 |
|------|---------|---------|
| 伪回归 | 独立非平稳序列的 OLS 可能"显著" | 直接回归价格是无意义的——必须差分或检验协整 |
| 协整 | $y_t - \beta x_t$ 平稳 | "各自乱走但不会分道扬镳"——配对交易的数学前提 |
| EG 两步法 | OLS + 残差 ADF | 两个序列协整的标准检验 |
| ECM | $\Delta y_t = \alpha z_{t-1} + \cdots$ ($\alpha<0$) | 短期波动向长期均衡修正的速度 |
| 配对交易 | 做多价差均值回复 | 市场中性策略——方向性风险被对冲 |

**从本章走向下一章**:
- 第23章将进入**随机分析**——从离散的 ARIMA/GARCH/协整走向连续的布朗运动和随机微分方程。如果说本阶段的主题是"用过去的统计结构预测未来", 第六阶段的主题将是"用随机过程理论为衍生品定价"。

---

## 22.9 练习题

### 数学推导

**题1——协整与伪回归的区别**:

(a) 两个独立的随机游走 $X_t$ 和 $Y_t$ 的回归残差是否有单位根？为什么？

(b) 如果 $Y_t = \beta X_t + \varepsilon_t$, 且 $\varepsilon_t \sim I(0)$, 回归 $Y_t$ 对 $X_t$ 的残差是否有单位根？用 Engle-Granger 检验的逻辑解释。

**题2——ECM 的推导**:

(a) 从协整关系 $y_t = \beta x_t + z_t$ ($z_t \sim I(0)$) 出发, 推导 ECM 表示: $\Delta y_t = \alpha z_{t-1} + \gamma \Delta x_t + \varepsilon_t$。 (提示: 取一阶差分, 用 $z_t = \rho z_{t-1} + \varepsilon_t$ 表示平稳 AR(1) 残差。)

(b) 验证: $\alpha < 0$ 是均值回复的必要条件。如果 $\alpha > 0$, 价差会如何表现？

**题3——对冲比率的稳定性**:

(a) 对冲比率 $\beta$ 在形成期和交易期是否可能不同？什么情况下 $\beta$ 会变化？这对配对交易策略有何影响？

(b) 如果用滚动窗口估计 $\beta$(如过去 252 个交易日), 与用全样本估计相比, 哪种方法更适合实盘交易？为什么？

### 编程实践

**题4——多行业协整扫描**: 对 `stock_data_50` 中所有同行业的两两股票对运行 Engle-Granger 检验。按行业分组统计协整对的比例——哪个行业的协整对最多？解释为什么有些行业(如银行)更容易出现协整。

**题5——配对交易的参数敏感性**: 基于 22.6 的策略框架, 测试不同参数组合: 阈值 ($\pm 1.5\sigma, \pm 2\sigma, \pm 2.5\sigma$)、形成期长度(1年/2年/3年)。哪个组合的夏普比率最高？参数选择的稳定性如何(不同时期的排名是否一致)？

---

## 22.10 参考文献

1. **Engle, R. F., & Granger, C. W. J.** (1987). Co-integration and error correction: representation, estimation, and testing. *Econometrica*, 55(2), 251-276. （协整理论的奠基之作——Engle-Granger 两步法和 ECM 的提出, 两人因此获 2003 年诺贝尔经济学奖）

2. **Johansen, S.** (1991). Estimation and hypothesis testing of cointegration vectors in Gaussian vector autoregressive models. *Econometrica*, 59(6), 1551-1580. （Johansen 检验——多变量协整的系统框架）

3. **Vidyamurthy, G.** (2004). *Pairs Trading: Quantitative Methods and Analysis*. Wiley. （配对交易的经典实务书籍——第3-5章系统讲解了协整在实际交易中的应用和陷阱）

4. **Granger, C. W. J., & Newbold, P.** (1974). Spurious regressions in econometrics. *Journal of Econometrics*, 2(2), 111-120. （伪回归问题的经典论文——模拟实验的细节和预警至今有效）

5. **Tsay, R. S.** (2010). *Analysis of Financial Time Series* (3rd ed.). Wiley. （第8章系统讲解协整检验、ECM 和多元时间序列建模）

---

> **愿我们都能在数字与代码之间, 找到理解市场的那把钥匙.**
>
> *数学的理解没有捷径, 量化的能力无法外包.*